# **Algoritmo de Dijkstra**

## Definição e Objetivos

**Definição:** O algoritmo de Dijkstra é um algoritmo usado para encontrar os menores caminhos possíveis entre todos os nós de um grafo ponderado. O algoritmo foi idealizado e concebido pelo cientista da computação [Esdger W. Dijkstra](https://en.wikipedia.org/wiki/Edsger_W._Dijkstra) em 1956 e publicado 3 anos posteriormente.

**Objetivos:** O objetivo do algoritmo é a otimização de rotas que podem ser representadas por por gráficos ponderados, como rotas em protocolos de rede [IS-IS](https://en.wikipedia.org/wiki/IS-IS)(Intermediate System to Intermediate System) e [OSPF](https://en.wikipedia.org/wiki/Open_Shortest_Path_First)(Open Shortest Path First).

## Potencialidades e Limitações

**Potencialidades:** As potencialidades desse algoritmo se dá justamente nas suas amplas implementações em problemas do mundo real. Como o roteamento de pacotes, navgeação GPS e inteligência artificial(path-finding para NPCs em jogos 2D). Além disso, o algoritmo assegura uma garantia matemática de que um `caminho ótimo` será encontrado caso não existam arestas negativas.

__Limitações:__ O algoritmo de Dijkstra só funciona em Grafos Acíclicos Direcionados (Dags). A principal limitação, como já discutido anteriormente, é o fato de que o algoritmo não funciona com arestas de peso negativo, para esses casos, seria usado o algoritmo de `Bellman-Ford`. Ademais, o algoritmo pode ser lento em grafos extremamente grandes caso seja implementado de uma forma não otimizada (Usando um Queue ou Heaps)

## Código e Visualização

Primeiro, vamos começar com uma visualização Simples do problema, para que o algoritmo seja mais didático:

<div>
<img src = "resources\grafo_exemplo.png" width = 500/>
</div>

Nossa intenção é chegar do ínicio até o fim da maneira com o __menor custo possível__. Para isso, precisaremos de 3 *Tabelas Hash (Dicionários)*, onde as tabelas de custos e dos pais será atualizada ao longo da realização do algoritmo, para isso, antes precisamos criar a representação do grafo.

In [1]:
grafo = {}
grafo["Inicio"] = {}
grafo["Inicio"]["A"] = 6
grafo["Inicio"]["B"] = 2
grafo["A"] = {}
grafo["A"]["Final"] = 1
grafo["B"] = {}
grafo["B"]["A"] = 3
grafo["B"]["Final"] = 5
grafo["Final"] = {}

<div>
<img src = "resources\hash_tables.png" width = 500/>
</div>

O código acima é apenas a criação do nosso grafo a partir de uma tabela hash. Perceba que, para armazenar os pesos, usamos tabelas hash aninhadas. Agora, vamos criar a tabela para os custos e a tabela para os pais, lembrando que vamos atualizar essa tabela gradativamente ao decorrer da execução do algoritmo de Dijkstra

In [2]:
# Representando o infinito
infinito = float("inf")

# Criando a tabela Hash de custos
custos = {}
custos["A"] = 6
custos["B"] = 2
custos["Final"] = infinito

# Criando a tabela Hash de pais
pais = {}
pais["A"] = "Inicio"
pais["B"] = "Inicio"
pais["Final"] = None

Por fim é necessário um array para manter registro de todos os vértices processados, pois eles não precisam ser processados mais de uma vez:

In [3]:
processados = []

Essa é **toda** a configuração necessária, agora podemos partir para o **algoritmo**, que funciona seguindo o fluxograma a seguir:

<div>
<img src = "resources\fluxograma.png" width = 500/>
</div>

In [4]:
def achar_menor_custo(custos: dict):
    menor_custo = infinito # float("inf")
    node_menor_custo = None
    for node in custos:
        custo = custos[node]
        if custo < menor_custo and node not in processados:
            menor_custo = custo
            node_menor_custo = node
    return node_menor_custo

In [5]:
def simple_dijkstra(grafo: dict, custos: dict, pais: dict):
    # O nó escolhido inicialmente é o nó com o menor custo, usando uma função específica para isso.
    node = achar_menor_custo(custos)
    
    # Enqunato existirem nós
    while node is not None:
        custo = custos[node]
        vizinhos = grafo[node]
        for n in vizinhos.keys():
            novo_custo = custo + vizinhos[n]
            if novo_custo < custos[n]:
                custos[n] = novo_custo
                pais[n] = node
        # Adicionando o Nó atual a lista de processados para que não passemos sobre ele
        processados.append(node)
        
        # escolhendo o próximo nó ao final de cada passo da iteração
        node = achar_menor_custo(custos)
        
    return custos, pais

Vamos testar nosso algoritmo!

In [6]:
novos_custos, novos_pais = simple_dijkstra(grafo=grafo, custos=custos, pais=pais)

print(f"Novos custos: {novos_custos} \nNovos pais: {novos_pais}")

Novos custos: {'A': 5, 'B': 2, 'Final': 6} 
Novos pais: {'A': 'B', 'B': 'Inicio', 'Final': 'A'}


**Os resultados fazem sentido!** De acordo com o nosso algoritmo, o caminho `ótimo` para atravessar esse grafo seria do `Início para B`, depois de `B para A`, e de `A para o final`. Além disso, o menor custo possível para ir do início até o final é **6**, o que também foi mostrado pelo algoritmo.

<div>
<img src = "resources\resultado_algoritmo.png" width = 500/>
</div>

Vamos fazer uma versão **Interativa** do algortimo para ver a funcionalidade na prática!
OBS.: Para a criação do menu interativo que permite a visualização, foi usada I.A generativa.

In [7]:
# Vamos alterar levemente a nossa função dijkstra original:
import copy

def achar_menor_custo(custos, processados):
    menor_custo = float("inf")
    nodo_menor_custo = None
    for nodo in custos:
        custo = custos[nodo]
        if custo < menor_custo and nodo not in processados:
            menor_custo = custo
            nodo_menor_custo = nodo
    return nodo_menor_custo

def dijkstra_com_historico(grafo, custos, pais):
    processados = []
    historico = [] # Nossa "câmera fotográfica"
    
    # Foto 0: O estado inicial antes de começar
    historico.append({
        "passo": 0,
        "nodo_atual": "Início",
        "custos": copy.deepcopy(custos),
        "pais": copy.deepcopy(pais),
        "processados": list(processados)
    })
    
    node = achar_menor_custo(custos, processados)
    contador_passo = 1
    
    while node is not None:
        custo = custos[node]
        vizinhos = grafo[node]
        
        for n in vizinhos.keys():
            novo_custo = custo + vizinhos[n]
            if novo_custo < custos[n]: 
                custos[n] = novo_custo
                pais[n] = node
                
        processados.append(node)
        
        # Tiramos uma foto do estado APÓS processar o nó atual
        historico.append({
            "passo": contador_passo,
            "nodo_atual": node,
            "custos": copy.deepcopy(custos),
            "pais": copy.deepcopy(pais),
            "processados": list(processados)
        })
        
        node = achar_menor_custo(custos, processados)
        contador_passo += 1
        
    return historico

Como Mencionado anteriormente, para uma melhor visualização do funcionamento do algoritmo, usei `I.A generativa` para construir uma **interface gráfica interativa** com o usuário. Acredito que o uso da I.A nesse caso se justifica pois anteriormente já havia implementado o algoritmo "na marra".

In [ ]:
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, widgets, HBox, VBox, HTML, Layout
from IPython.display import display, clear_output, HTML as IPythonHTML
import copy

# ============================================
# VERSÃO MODERNIZADA DO INTERFACE
# ============================================

# Definindo os dados (exatamente como antes)
grafo = {
    "inicio": {"a": 6, "b": 2},
    "a": {"fim": 1},
    "b": {"a": 3, "fim": 5},
    "fim": {}
}
infinito = float("inf")
custos = {"a": 6, "b": 2, "fim": infinito}
pais = {"a": "inicio", "b": "inicio", "fim": None}

# rodando a função para pegar os dados
historico_execucao = dijkstra_com_historico(grafo, custos, pais)

# Estilos CSS modernos (injetados uma única vez)
style_css = """
<style>
.dijkstra-card {
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    border-radius: 15px;
    padding: 20px;
    margin: 10px 0;
    color: white;
    box-shadow: 0 4px 15px rgba(0,0,0,0.2);
}
.dijkstra-step {
    background: #f8f9fa;
    border-left: 4px solid #667eea;
    padding: 15px;
    margin: 8px 0;
    border-radius: 0 10px 10px 0;
}
.dijkstra-header {
    font-size: 24px;
    font-weight: bold;
    text-align: center;
    margin-bottom: 20px;
    color: #2c3e50;
}
.cost-badge {
    background: #27ae60;
    color: white;
    padding: 5px 12px;
    border-radius: 20px;
    font-weight: bold;
    display: inline-block;
    margin: 2px;
}
.parent-badge {
    background: #3498db;
    color: white;
    padding: 5px 12px;
    border-radius: 20px;
    display: inline-block;
    margin: 2px;
}
.processed-badge {
    background: #e74c3c;
    color: white;
    padding: 5px 12px;
    border-radius: 20px;
    display: inline-block;
    margin: 2px;
}
.current-node {
    background: linear-gradient(135deg, #f093fb 0%, #f5576c 100%);
    color: white;
    padding: 10px 20px;
    border-radius: 25px;
    font-size: 18px;
    font-weight: bold;
    text-align: center;
    box-shadow: 0 4px 10px rgba(245, 87, 108, 0.4);
}
.slider-container {
    background: #ffffff;
    border-radius: 15px;
    padding: 20px;
    box-shadow: 0 2px 20px rgba(0,0,0,0.1);
    margin: 10px 0;
}
</style>
"""

# Injetar CSS uma vez
display(IPythonHTML(style_css))

# Criar widget de saída HTML (sem Output widget)
output_html = widgets.HTML()

def criar_interface_moderna(passo):
    estado = historico_execucao[passo]
    
    # Header com gradiente
    header_html = f"""
    <div class="dijkstra-card">
        <div style="font-size: 28px; font-weight: bold; text-align: center;">
            🔍 Algoritmo de Dijkstra - Iteração {estado['passo']}
        </div>
        <div style="text-align: center; margin-top: 10px;">
            <span style="background: rgba(255,255,255,0.3); padding: 8px 20px; border-radius: 20px;">
                📍 Nó Atual: <strong>{estado['nodo_atual'].upper()}</strong>
            </span>
        </div>
    </div>
    """
    
    # Container com sombra
    container_html = f"""
    <div class="slider-container">
        <div style="display: flex; justify-content: space-between; margin-bottom: 15px;">
            <div style="flex: 1; margin-right: 10px;">
                <h4 style="color: #27ae60; margin-bottom: 10px;">💰 Custos Atuais</h4>
                <div style="background: #f0f0f0; padding: 15px; border-radius: 10px;">
                    {''.join([f'<span class="cost-badge">{no}: {custo if custo != float("inf") else "∞"}</span>' for no, custo in estado['custos'].items()])}
                </div>
            </div>
            <div style="flex: 1; margin-left: 10px;">
                <h4 style="color: #3498db; margin-bottom: 10px;">👨‍👩‍👧 Pais</h4>
                <div style="background: #f0f0f0; padding: 15px; border-radius: 10px;">
                    {''.join([f'<span class="parent-badge">{no}: {pai if pai else "None"}</span>' for no, pai in estado['pais'].items()])}
                </div>
            </div>
        </div>
        <div>
            <h4 style="color: #e74c3c; margin-bottom: 10px;">✅ Nós Processados</h4>
            <div style="background: #f0f0f0; padding: 15px; border-radius: 10px;">
                {''.join([f'<span class="processed-badge">{n}</span>' for n in estado['processados']]) if estado['processados'] else '<span style="color: #999;">Nenhum nó processado ainda</span>'}
            </div>
        </div>
    </div>
    """
    
    # Barra de progresso visual
    total_passos = len(historico_execucao) - 1
    progresso = (passo / total_passos) * 100 if total_passos > 0 else 0
    
    progress_html = f"""
    <div style="margin-top: 15px;">
        <div style="display: flex; justify-content: space-between; color: #666; margin-bottom: 5px;">
            <span>Progresso</span>
            <span>{passo}/{total_passos}</span>
        </div>
        <div style="background: #e0e0e0; border-radius: 10px; height: 20px; overflow: hidden;">
            <div style="background: linear-gradient(90deg, #667eea, #764ba2); width: {progresso}%; height: 100%; border-radius: 10px; transition: width 0.3s ease;"></div>
        </div>
    </div>
    """
    
    # Explicação do passo atual
    if passo == 0:
        explicacao = "📋 Estado inicial: Os custos representam a distância direta do nó de início."
    elif estado['nodo_atual'] == 'b':
        explicacao = "🔄 Processando nó 'B' (custo=2): Verificamos seus vizinhos e atualizamos custos se encontrarmos caminhos menores."
    elif estado['nodo_atual'] == 'a':
        explicacao = "🔄 Processando nó 'A' (custo=6): Atualizamos o custo para o nó final através de A."
    elif estado['nodo_atual'] == 'fim':
        explicacao = "🏁 Processando nó 'Final': Último nó processado. Algoritmo concluído!"
    else:
        explicacao = f"➡️ Processando nó '{estado['nodo_atual']}'..."
        
    explicacao_html = f"""
    <div style="margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 10px; border-left: 4px solid #ffc107;">
        {explicacao}
    </div>
    """
    
    # Combinar tudo em um único HTML
    output_html.value = header_html + container_html + progress_html + explicacao_html

# Criar slider estilizado
max_passos = len(historico_execucao) - 1

# Slider personalizado
slider = widgets.IntSlider(
    min=0, 
    max=max_passos, 
    step=1, 
    value=0, 
    description='',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='80%')
)

# Label do slider
slider_label = widgets.HTML(
    value="<div style='text-align: center; padding: 15px; font-size: 16px; color: #2c3e50;'><strong>Arraste o slider para visualizar cada etapa do algoritmo</strong></div>"
)

# Layout final - slider em cima, output embaixo (sem usar interact)
interface_layout = VBox([
    widgets.HTML("<h2 style='color: #2c3e50; text-align: center; margin-bottom: 10px;'>🎯 Visualizador Interativo de Dijkstra</h2>"),
    slider_label,
    slider,
    output_html
], layout=Layout(padding='20px', align_items='center'))

# Exibir a interface completa
display(interface_layout)

# Vincular o slider à função usando observe
def on_slider_change(change):
    criar_interface_moderna(change['new'])

slider.observe(on_slider_change, names='value')

# Inicializar o primeiro passo
criar_interface_moderna(0)

## Fontes

 - [WikiPedia](https://en.wikipedia.org/wiki/Dijkstra%27s_algorithm)
 - [GeeksForGeeks](https://www.w3schools.com/dsa/dsa_algo_graphs_dijkstra.php)
 - Visualização Rápida no [YouTube](https://www.youtube.com/watch?v=EFg3u_E6eHU&t=439s)
 - Mais uma visualização no [YouTube](https://www.youtube.com/watch?v=GazC3A4OQTE)
 - Entendendo Algoritmos: [Link](https://www.amazon.com.br/Entendendo-Algoritmos-Ilustrado-Programadores-Curiosos/dp/8575225634)